# Dataset-Preprocessing withIP

--- 
**Note**: 
This project uses data from https://www.kaggle.com/datasets/shriyashjagtap/fraudulent-e-commerce-transactions






#### Columns in Original Dataset:
    Transaction ID,
    Customer ID,
    Transaction Amount,
    Transaction Date,
    Payment Method,
    Product Category,
    Quantity,
    Customer Age,
    Customer Location,
    Device Used,
    IP Address,
    Shipping Address,
    Billing Address,
    Is Fraudulent,
    Account Age Days,
    Transaction Hour

## **Objective:**
### Clean raw data from kaggle and save processed arrays for all model notebooks to share 





1. Deciding what data to use

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/Fraudulent_E-Commerce_Transaction_Data.csv')

df = pd.read_csv(file_path)

print(f"Original shape: {df.shape}")

Original shape: (1472952, 16)


In [3]:
# Check for duplicates in IP Address
if df['IP Address'].duplicated().any():
    print("IP Address has repeated values. It may be useful for modeling.")
    shared_ips = (df['IP Address'].value_counts() > 1).sum()
    print(f"Number of unique IPs used in multiple transactions: {shared_ips}")

# Check for duplicates in CustomerID
if df['Customer ID'].duplicated().any():
    print("CustomerID has repeated values. It may be useful for modeling.")
else:
    print("CustomerID has no repeated values. It can be dropped as it does not add value to the model.")

IP Address has repeated values. It may be useful for modeling.
Number of unique IPs used in multiple transactions: 301
CustomerID has no repeated values. It can be dropped as it does not add value to the model.


In [4]:
# 1. Address Match Binary Feature
df['Address Match'] = (df['Shipping Address'] == df['Billing Address']).astype(int)

# 2. Cyclical Encoding for Time
df['Hour_Sin'] = np.sin(2 * np.pi * df['Transaction Hour'] / 24)
df['Hour_Cos'] = np.cos(2 * np.pi * df['Transaction Hour'] / 24)

# 3. One-Hot Encoding (drop_first=True to avoid multicollinearity)
categorical_features = ['Payment Method', 'Product Category', 'Device Used']
df = pd.get_dummies(df, columns=categorical_features, drop_first=True)

In [5]:
# 1. Sort by Date
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])
df = df.sort_values(by='Transaction Date').reset_index(drop=True)

# 2. IP Velocity: Transactions from this IP in the last 1 hour
df_time = df.set_index('Transaction Date')
df['IP_Transaction_Count_1H'] = df_time.groupby('IP Address')['Transaction ID'].rolling('1h').count().values

# 3. Linkage: Total Unique Customers per IP
ip_user_counts = df.groupby('IP Address')['Customer ID'].nunique().reset_index()
ip_user_counts.rename(columns={'Customer ID': 'Unique_Customers_Per_IP'}, inplace=True)
df = df.merge(ip_user_counts, on='IP Address', how='left')

# Restore index
df = df.reset_index(drop=True)

In [6]:
# 1. Drop all remaining raw, high-cardinality, or redundant columns
cols_to_drop = [
    'Transaction ID', 'Customer ID', 'IP Address', 
    'Shipping Address', 'Billing Address', 'Transaction Hour', 
    'Transaction Date', 'Customer Location'
]
# Only drop columns that actually exist in the dataframe to avoid errors
existing_cols_to_drop = [col for col in cols_to_drop if col in df.columns]
df.drop(columns=existing_cols_to_drop, inplace=True)

# 2. Separate Features and Label
X = df.drop(columns=['Is Fraudulent'])
y = df['Is Fraudulent']

print(f"Final Feature Matrix Shape: {X.shape}")

Final Feature Matrix Shape: (1472952, 18)


In [7]:
# 1. Print a list of all columns
print(f"Total features in use: {len(df.columns)}")
print("-" * 30)
for col in df.columns:
    print(f"- {col}")

print("\n") 

# 2. Print the detailed info (includes data types and non-null counts)
print("Detailed DataFrame Information:")
print("-" * 30)
df.info()

Total features in use: 19
------------------------------
- Transaction Amount
- Quantity
- Customer Age
- Is Fraudulent
- Account Age Days
- Address Match
- Hour_Sin
- Hour_Cos
- Payment Method_bank transfer
- Payment Method_credit card
- Payment Method_debit card
- Product Category_electronics
- Product Category_health & beauty
- Product Category_home & garden
- Product Category_toys & games
- Device Used_mobile
- Device Used_tablet
- IP_Transaction_Count_1H
- Unique_Customers_Per_IP


Detailed DataFrame Information:
------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 1472952 entries, 0 to 1472951
Data columns (total 19 columns):
 #   Column                            Non-Null Count    Dtype  
---  ------                            --------------    -----  
 0   Transaction Amount                1472952 non-null  float64
 1   Quantity                          1472952 non-null  int64  
 2   Customer Age                      1472952 non-null  int64  
 3   Is Fraudulent

### 2. Data Splitting

- Train (70%) — normal transactions only
- Validation (15%) — mix of normal + fraud
- Test (15%) — mix of normal + fraud

In [8]:
from sklearn.model_selection import train_test_split

normal_data = df[df['Is Fraudulent'] == 0].drop(columns=['Is Fraudulent'])
fraud_data = df[df['Is Fraudulent'] == 1].drop(columns=['Is Fraudulent'])


""" 

Splitting the data into training, validation, and testing sets
Then combine the normal and fraudulent data for val and test

random_state is set to shuffle the data bfr splitting, 
    = 42 is set for reproducibility.


"""
# 30% set for testing, 70% training
normal_train, norm_temp = train_test_split(normal_data, test_size=0.3, random_state=42) 
# split the 30% into 15% validation and 15% testing
normal_val, normal_test = train_test_split(norm_temp, test_size=0.5, random_state=42) 

fraud_val, fraud_test = train_test_split(fraud_data, test_size=0.5, random_state=42)

# Combine the normal and fraudulent data for val and test
val_df = pd.concat([normal_val.assign(label=0), fraud_val.assign(label=1)]).sample(frac=1, random_state=42)
test_df = pd.concat([normal_test.assign(label=0), fraud_test.assign(label=1)]).sample(frac=1, random_state=42)

x_train = normal_train.values.astype('float32') # Exclude labels column (everything is same value) 
x_val = val_df.drop(columns=['label']).values.astype('float32') # drop label as label is the target variable
y_val = val_df['label'].values # target variable for val
x_test = test_df.drop(columns=['label']).values.astype('float32')
y_test = test_df['label'].values

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")
print(f"Val fraud ratio:  {y_val.mean():.2%}")
print(f"Test fraud ratio: {y_test.mean():.2%}")

Train: (979379, 18), Val: (246786, 18), Test: (246787, 18)
Val fraud ratio:  14.96%
Test fraud ratio: 14.96%


In [10]:
# Standardize the features using StandardScaler

from sklearn.preprocessing import StandardScaler
import joblib
import os

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler_withIP.pkl')


['../models/scaler_withIP.pkl']

### 3. Save processed arrays to disk

Run this once. All other notebooks load from these files via `load_processed()`.

In [11]:
# Save processed arrays so all other notebooks can load them instantly
import numpy as np, os, pickle

os.makedirs("../data/processed_withIP", exist_ok=True)
np.savez_compressed("../data/processed_withIP/train.npz", x_train=x_train)
np.savez_compressed("../data/processed_withIP/val.npz",   x_val=x_val,   y_val=y_val)
np.savez_compressed("../data/processed_withIP/test.npz",  x_test=x_test, y_test=y_test)

# Save feature names for downstream notebooks
feature_names = [c for c in df.columns if c != "Is Fraudulent"]
schema = {"feature_names": feature_names, "input_dim": int(x_train.shape[1])}
os.makedirs("../models", exist_ok=True)
with open("../models/feature_schema_withIP.pkl", "wb") as f:
    pickle.dump(schema, f)

print(f"Saved train:  {x_train.shape}")
print(f"Saved val:    {x_val.shape}  (fraud rate: {y_val.mean():.2%})")
print(f"Saved test:   {x_test.shape}  (fraud rate: {y_test.mean():.2%})")
print(f"Saved schema: {len(feature_names)} features -> ../models/feature_schema_withIP.pkl")
print("Done. All other notebooks can now call load_processed().")


Saved train:  (979379, 18)
Saved val:    (246786, 18)  (fraud rate: 14.96%)
Saved test:   (246787, 18)  (fraud rate: 14.96%)
Saved schema: 18 features -> ../models/feature_schema_withIP.pkl
Done. All other notebooks can now call load_processed().
